# 04 — Stage 2: CodeBERT NLP Branch

**Project:** HierFuse — Hierarchical PowerShell / LotL Detection with Learned Fusion  
**Stage:** 2 of 3 — semantic encoder (text modality)  
**Platform:** Kaggle Free Tier · **P100 GPU recommended** (T4 also works)  
**Runtime:** ~3–4 hours (3 seeds × ~1.5 h each)  
**Inputs:** `02-splits-dataset`  
**Outputs:** `/kaggle/working/nlp/`

## Architecture

`microsoft/codebert-base` (125M params) fine-tuned for binary PS/LotL classification.

- **Sliding-window inference** (stride=128, max 8 windows) for scripts >512 tokens
- **Probability aggregation:** `max` across windows ("flag if any chunk is malicious")
- **Embedding aggregation:** `mean` across windows (stable 768-d CLS vector for fusion)
- Returns: per-script probability + 768-d CLS embedding → Stage 3 fusion

**DistilBERT ablation:** Set `MODEL_NAME = "distilbert-base-uncased"` for the  
cheap baseline in Table 1.

## Outputs
- `checkpoints/codebert_seed{N}/` — fine-tuned model
- `probs/{train,val,test,test_c2}_probs_seed{N}.npy`
- `embeddings/{train,val,test,test_c2}_emb_seed{N}.npy`
- `nlp_results.json` — all metrics


## Cell 1 — Bootstrap: locate datasets, create output dirs

In [1]:
import os, sys, json, time, gc, platform, warnings, random
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID  : {RUN_ID}')
print(f'Python  : {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')

def _rglob_first(root, pattern):
    for m in Path(root).rglob(pattern): return m
    return None

KAGGLE_INPUT = Path('/kaggle/input')

SPLITS_DIR = None
for m in KAGGLE_INPUT.rglob('train_seed42.parquet'):
    SPLITS_DIR = m.parent; break

WEIGHTS_PATH = _rglob_first(KAGGLE_INPUT, 'class_weights_per_seed.json')
LOTL_PARQUET   = _rglob_first(KAGGLE_INPUT, 'lotl_mimicry_dataset.parquet')
BENIGN_PARQUET = _rglob_first(KAGGLE_INPUT, 'benign_fp_stress_dataset.parquet')

print()
for label, path in [('Splits dir', SPLITS_DIR), ('Weights', WEIGHTS_PATH),
                     ('LotL parquet (optional)', LOTL_PARQUET),
                     ('Benign FP parquet (optional)', BENIGN_PARQUET)]:
    ok = path is not None and Path(path).exists()
    print(f'  {"[OK]" if ok else "[MISSING]":<12} {label}: {path}')

if not SPLITS_DIR or not WEIGHTS_PATH:
    raise FileNotFoundError('Attach 02-splits-dataset before running this notebook.')

OUT_ROOT      = Path('/kaggle/working/nlp')
MODELS_DIR    = OUT_ROOT / 'checkpoints'
PROBS_DIR     = OUT_ROOT / 'probs'
EMBEDDINGS_DIR= OUT_ROOT / 'embeddings'
for d in [MODELS_DIR, PROBS_DIR, EMBEDDINGS_DIR]: d.mkdir(parents=True, exist_ok=True)

with open(WEIGHTS_PATH) as f: CLASS_WEIGHTS = json.load(f)
print(f'\nOutputs: {OUT_ROOT}')
print('Cell 1 OK.')


Run ID  : 20260529_070646
Python  : 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35

  [OK]         Splits dir: /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits
  [OK]         Weights: /kaggle/input/datasets/onkarkmane1501/02-splits-dataset/splits/class_weights_per_seed.json
  [MISSING]    LotL parquet (optional): None
  [MISSING]    Benign FP parquet (optional): None

Outputs: /kaggle/working/nlp
Cell 1 OK.


## Cell 2 — Hyperparameters

In [2]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/codebert-base"   # swap to "distilbert-base-uncased" for ablation

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS          = 2
TRAIN_BATCH_SIZE    = 8
EVAL_BATCH_SIZE     = 16
GRAD_ACCUM_STEPS    = 4        # effective batch = 8 × 4 = 32
LEARNING_RATE       = 2e-5
WEIGHT_DECAY        = 0.01
MAX_SEQ_LEN         = 512
SLIDING_STRIDE      = 128
MAX_WINDOWS         = 8
WARMUP_RATIO        = 0.06
FP16                = True     # T4/P100 — disable if OOM

SEEDS                  = [42, 1337, 2024]
PER_SEED_TIME_BUDGET_S = 7_200  # 2 hours; gracefully skips epoch 2 if exceeded

GLOBAL_T0 = time.time()

print('Hyperparameters:')
for k,v in dict(MODEL_NAME=MODEL_NAME, EPOCHS=NUM_EPOCHS,
                EFFECTIVE_BATCH=TRAIN_BATCH_SIZE*GRAD_ACCUM_STEPS,
                LR=LEARNING_RATE, MAX_SEQ_LEN=MAX_SEQ_LEN,
                SLIDING_STRIDE=SLIDING_STRIDE, FP16=FP16).items():
    print(f'  {k:<28}: {v}')


Hyperparameters:
  MODEL_NAME                  : microsoft/codebert-base
  EPOCHS                      : 2
  EFFECTIVE_BATCH             : 32
  LR                          : 2e-05
  MAX_SEQ_LEN                 : 512
  SLIDING_STRIDE              : 128
  FP16                        : True


## Cell 3 — Installs and imports (pinned transformers version)

In [3]:
import subprocess
import sys

# Suppress the noisy pip dependency resolver errors by redirecting stderr
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--disable-pip-version-check',
    'transformers>=4.44,<5.0', 'datasets', 'accelerate', 'pyarrow',
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig, AutoModel, AutoTokenizer,
    get_linear_schedule_with_warmup,
    set_seed as hf_set_seed,
)
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
)
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('nlp')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log.info(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    log.info(f'GPU: {torch.cuda.get_device_name(0)}  '
             f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    log.warning('Running on CPU — expect ~6× slower training.')

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
log.info(f'Tokenizer: {MODEL_NAME}  vocab={TOKENIZER.vocab_size:,}')
print('Cell 3 OK.')


07:07:17 | INFO | Device: cuda
07:07:17 | INFO | GPU: Tesla T4  VRAM: 15.6 GB


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

07:07:19 | INFO | Tokenizer: microsoft/codebert-base  vocab=50,265


Cell 3 OK.


## Cell 4 — Load splits helper

In [4]:
def load_seed_data(seed):
    """Load all 4 splits for a given seed."""
    log.info(f'  Loading splits for seed {seed}...')
    splits = {}
    for split in ['train','val','test','test_c2']:
        df = pd.read_parquet(SPLITS_DIR/f'{split}_seed{seed}.parquet')
        splits[split] = {'texts': df['script_text'].tolist(), 'labels': df['label'].values}
        log.info(f'    {split:8s}: {len(df):,} rows '
                 f'(mal={int(df.label.sum())}, ben={int((df.label==0).sum())})')
    return splits

print('Cell 4 OK.')


Cell 4 OK.


## Cell 5 — PyTorch Dataset and CodeBERT classifier

In [5]:
class PSDataset(Dataset):
    """Truncates at MAX_SEQ_LEN; sliding window applied only at inference."""
    def __init__(self, texts, labels, max_len=MAX_SEQ_LEN):
        self.texts = texts; self.labels = labels; self.max_len = max_len
        
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        enc = TOKENIZER(self.texts[idx], max_length=self.max_len,
                        truncation=True, padding='max_length', return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         torch.tensor(self.labels[idx], dtype=torch.long)}


class CodeBertClassifier(nn.Module):
    """CodeBERT + single-logit binary head.
    forward() returns dict with keys: loss (optional), logits [B], embedding [B,768]."""
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.config     = AutoConfig.from_pretrained(model_name)
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.config.hidden_size      # 768
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask, labels=None, pos_weight=None, return_embedding=False):
        out        = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls_hidden = out.last_hidden_state[:, 0, :]    # CLS token [B, 768]
        logits     = self.classifier(self.dropout(cls_hidden)).squeeze(-1)  # [B]
        loss = None
        if labels is not None:
            pw        = torch.tensor([pos_weight], device=logits.device, dtype=logits.dtype) if pos_weight else None
            criterion = nn.BCEWithLogitsLoss(pos_weight=pw) if pw is not None else nn.BCEWithLogitsLoss()
            loss      = criterion(logits, labels.float())
        if return_embedding:
            return {'loss': loss, 'logits': logits, 'embedding': cls_hidden}
        return {'loss': loss, 'logits': logits}

print('Dataset and CodeBertClassifier defined.')


Dataset and CodeBertClassifier defined.


## Cell 6 — Sliding-window inference

In [6]:
@torch.no_grad()
def sliding_window_inference(model, texts, batch_size=EVAL_BATCH_SIZE,
                              max_len=MAX_SEQ_LEN, stride=SLIDING_STRIDE,
                              max_windows=MAX_WINDOWS, desc='infer'):
    """Sliding-window inference over a list of texts.
    Probs: max across windows (any-chunk-fires rule).
    Embeddings: mean across windows (stable 768-d for fusion).
    Returns: (probs np.float32 [N], embs np.float32 [N,768])
    """
    model.eval()
    N = len(texts)
    window_ids, window_masks, script_of_window = [], [], []

    for i, text in enumerate(texts):
        enc = TOKENIZER(text, truncation=False, padding=False, return_tensors=None)
        ids = enc['input_ids']
        if len(ids) <= max_len:
            windows = [ids]
        else:
            windows = []
            for start in range(0, len(ids)-1, stride):
                windows.append(ids[start:start+max_len])
                if len(windows) >= max_windows: break
        for win in windows:
            padded  = win[:max_len] + [TOKENIZER.pad_token_id] * max(0, max_len - len(win))
            mask    = [1]*min(len(win), max_len) + [0]*max(0, max_len - len(win))
            window_ids.append(padded[:max_len]); window_masks.append(mask[:max_len])
            script_of_window.append(i)

    # Batch inference
    all_logits = []
    all_embs   = []
    for start in range(0, len(window_ids), batch_size):
        end  = min(start+batch_size, len(window_ids))
        ids  = torch.tensor(window_ids[start:end],  device=DEVICE)
        mask = torch.tensor(window_masks[start:end], device=DEVICE)
        out  = model(ids, mask, return_embedding=True)
        all_logits.extend(torch.sigmoid(out['logits']).cpu().numpy().tolist())
        all_embs.extend(out['embedding'].cpu().numpy())

    # Aggregate per-script: probs=max, embs=mean
    probs = np.zeros(N, dtype=np.float32)
    embs  = np.zeros((N, 768), dtype=np.float32)
    counts= np.zeros(N, dtype=np.int32)
    for idx, (logit, emb) in enumerate(zip(all_logits, all_embs)):
        sci = script_of_window[idx]
        probs[sci] = max(probs[sci], float(logit))
        embs[sci] += np.array(emb, dtype=np.float32)
        counts[sci] += 1
    embs /= np.maximum(counts[:,None], 1)
    return probs, embs


def evaluate_probs(y_true, y_prob, label=''):
    y_pred = (y_prob >= 0.5).astype(int)
    f1     = float(f1_score(y_true, y_pred, zero_division=0))
    auroc  = float(roc_auc_score(y_true, y_prob))
    ap     = float(average_precision_score(y_true, y_prob))
    fpr_a, tpr_a, _ = roc_curve(y_true, y_prob)
    tpr1   = float(np.interp(0.01, fpr_a, tpr_a))
    result = {'f1':round(f1,4),'auroc':round(auroc,4),'pr_auc':round(ap,4),'tpr_at_1fpr':round(tpr1,4)}
    if label:
        log.info(f'  [{label}] F1={f1:.4f}  AUROC={auroc:.4f}  PR-AUC={ap:.4f}  TPR@1%FPR={tpr1:.4f}')
    return result

print('Inference helpers defined.')


Inference helpers defined.


## Cell 7 — Training function

In [7]:
def train_one_seed(seed, data, time_budget_s):
    """Fine-tune CodeBERT for one seed. Returns (model, elapsed_seconds)."""
    set_all_seeds(seed)
    t0 = time.time()

    w         = CLASS_WEIGHTS[str(seed)]
    pos_weight= float(w.get('pos_weight_bce', w.get('scale_pos_weight', 1.0)))

    train_ds = PSDataset(data['train']['texts'], data['train']['labels'])
    val_ds   = PSDataset(data['val']['texts'],   data['val']['labels'])
    train_ld = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_ld   = DataLoader(val_ds,   batch_size=EVAL_BATCH_SIZE,  shuffle=False,num_workers=2, pin_memory=True)

    model    = CodeBertClassifier().to(DEVICE)
    optimizer= torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    n_steps  = NUM_EPOCHS * len(train_ld) // GRAD_ACCUM_STEPS
    n_warmup = int(n_steps * WARMUP_RATIO)
    scheduler= get_linear_schedule_with_warmup(optimizer, n_warmup, n_steps)
    scaler   = torch.cuda.amp.GradScaler(enabled=(FP16 and DEVICE=='cuda'))

    best_val_f1  = 0.0
    ckpt_path    = MODELS_DIR / f'codebert_seed{seed}_best.pt'
    patience_cnt = 0
    PATIENCE     = 2

    for epoch in range(1, NUM_EPOCHS+1):
        if (time.time()-t0) > time_budget_s:
            log.warning(f'  Time budget exceeded — skipping epoch {epoch}'); break
        model.train()
        total_loss = 0.0; optimizer.zero_grad()
        for step, batch in enumerate(train_ld):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labs = batch['labels'].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(FP16 and DEVICE=='cuda')):
                out  = model(ids, mask, labels=labs, pos_weight=pos_weight)
                loss = out['loss'] / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step+1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step(); optimizer.zero_grad()
            total_loss += loss.item() * GRAD_ACCUM_STEPS

        # Quick val loop (no sliding window for speed)
        model.eval(); val_preds=[]; val_labs=[]
        with torch.no_grad():
            for batch in val_ld:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                out  = model(ids, mask)
                val_preds.extend(torch.sigmoid(out['logits']).cpu().numpy().tolist())
                val_labs.extend(batch['labels'].numpy().tolist())
        val_f1 = float(f1_score(val_labs, [1 if p>0.5 else 0 for p in val_preds], zero_division=0))
        avg_loss = total_loss / len(train_ld)
        log.info(f'  Epoch {epoch}/{NUM_EPOCHS}  loss={avg_loss:.4f}  val_F1={val_f1:.4f}  '
                 f'({(time.time()-t0)/60:.1f} min)')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_f1': val_f1},
                       ckpt_path)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE: log.info('  Early stopping.'); break

    # Load best checkpoint
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        log.info(f'  Best val F1: {best_val_f1:.4f} (epoch {ckpt["epoch"]})')

    return model, time.time()-t0


def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

print('Training function defined.')


Training function defined.


## Cell 8 — Main training + inference loop (3 seeds)

In [8]:
all_results = {}

for seed in SEEDS:
    log.info(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
    seed_t0 = time.time()

    data  = load_seed_data(seed)
    model, train_dur = train_one_seed(seed, data, PER_SEED_TIME_BUDGET_S)

    # Sliding-window inference on all splits
    log.info('  Running sliding-window inference...')
    tr_probs, tr_embs    = sliding_window_inference(model, data['train']['texts'],   desc='train')
    va_probs, va_embs    = sliding_window_inference(model, data['val']['texts'],     desc='val')
    te_probs, te_embs    = sliding_window_inference(model, data['test']['texts'],    desc='test')
    tec2_probs, tec2_embs= sliding_window_inference(model, data['test_c2']['texts'], desc='test_c2')

    # Evaluate
    tr_m   = evaluate_probs(data['train']['labels'],   tr_probs,   'train')
    va_m   = evaluate_probs(data['val']['labels'],     va_probs,   'val')
    te_m   = evaluate_probs(data['test']['labels'],    te_probs,   'balanced test')
    tec2_m = evaluate_probs(data['test_c2']['labels'], tec2_probs, 'C2 test')

    # Save prob arrays and embeddings
    for split_name, probs, embs in [
        ('train', tr_probs, tr_embs), ('val', va_probs, va_embs),
        ('test', te_probs, te_embs),  ('test_c2', tec2_probs, tec2_embs)
    ]:
        np.save(PROBS_DIR     / f'{split_name}_probs_seed{seed}.npy', probs)
        np.save(EMBEDDINGS_DIR/ f'{split_name}_emb_seed{seed}.npy',   embs)

    all_results[str(seed)] = {
        'train_metrics':   tr_m, 'val_metrics': va_m,
        'test_metrics':    te_m, 'test_c2_metrics': tec2_m,
        'best_val_f1':     round(float(max(va_probs >= 0.5)*0 + va_m['f1']), 4),
        'elapsed_min':     round((time.time()-seed_t0)/60, 1),
    }

    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

results_path = OUT_ROOT / 'nlp_results.json'
with open(results_path,'w') as f: json.dump(all_results, f, indent=2)
log.info(f'Results saved: {results_path}')
log.info(f'Total wall time: {(time.time()-GLOBAL_T0)/60:.1f} min')
print('Cell 8 OK.')


07:07:19 | INFO | 
SEED 42
07:07:19 | INFO |   Loading splits for seed 42...
07:07:24 | INFO |     train   : 12,176 rows (mal=6087, ben=6089)
07:07:25 | INFO |     val     : 2,610 rows (mal=1306, ben=1304)
07:07:26 | INFO |     test    : 2,610 rows (mal=1305, ben=1305)
07:07:26 | INFO |     test_c2 : 1,450 rows (mal=145, ben=1305)
2026-05-29 07:07:28.341544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780038448.515258      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780038448.565941      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780038448.988077      23 computation_placer.cc:177] computation placer already registered. 

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

07:23:52 | INFO |   Epoch 1/2  loss=0.4246  val_F1=0.8261  (16.2 min)
07:40:05 | INFO |   Epoch 2/2  loss=0.3013  val_F1=0.7956  (32.4 min)
07:40:06 | INFO |   Best val F1: 0.8261 (epoch 1)
07:40:06 | INFO |   Running sliding-window inference...
Token indices sequence length is longer than the specified maximum sequence length for this model (592 > 512). Running this sequence through the model will result in indexing errors
09:06:10 | INFO |   [train] F1=0.7652  AUROC=0.9470  PR-AUC=0.9546  TPR@1%FPR=0.6976
09:06:10 | INFO |   [val] F1=0.7585  AUROC=0.9326  PR-AUC=0.9446  TPR@1%FPR=0.6715
09:06:10 | INFO |   [balanced test] F1=0.7630  AUROC=0.9355  PR-AUC=0.9470  TPR@1%FPR=0.6912
09:06:10 | INFO |   [C2 test] F1=0.2707  AUROC=0.9231  PR-AUC=0.7823  TPR@1%FPR=0.6690
09:06:11 | INFO | 
SEED 1337
09:06:11 | INFO |   Loading splits for seed 1337...
09:06:17 | INFO |     train   : 12,176 rows (mal=6087, ben=6089)
09:06:18 | INFO |     val     : 2,610 rows (mal=1306, ben=1304)
09:06:19 | INF

Cell 8 OK.


## Cell 9 — Results table (Table 1, CodeBERT row)

In [9]:
done = [s for s in SEEDS if str(s) in all_results and 'test_metrics' in all_results[str(s)]]
if not done:
    print('No completed seeds — run Cell 8 first.')
else:
    METRICS  = [('f1','F1'),('auroc','AUROC'),('pr_auc','PR-AUC'),('tpr_at_1fpr','TPR@1%FPR')]
    VARIANTS = [('test_metrics','Balanced test'),('test_c2_metrics','C2 test (≤10% mal)')]
    for vkey, vlabel in VARIANTS:
        print(f'\n=== CodeBERT — {vlabel} ===')
        print(f'{"Metric":<14}  {"Mean ± Std":>20}')
        print('-'*36)
        for mkey, mname in METRICS:
            vals = [all_results[str(s)][vkey][mkey] for s in done]
            print(f'{mname:<14}  {np.mean(vals):.4f} ± {np.std(vals):.4f}')
    print()
    print('Embeddings (768-d CLS) saved to /kaggle/working/nlp/embeddings/')
    print('Probabilities saved to /kaggle/working/nlp/probs/')



=== CodeBERT — Balanced test ===
Metric                    Mean ± Std
------------------------------------
F1              0.7886 ± 0.0191
AUROC           0.9425 ± 0.0051
PR-AUC          0.9519 ± 0.0036
TPR@1%FPR       0.7009 ± 0.0165

=== CodeBERT — C2 test (≤10% mal) ===
Metric                    Mean ± Std
------------------------------------
F1              0.3009 ± 0.0236
AUROC           0.9344 ± 0.0080
PR-AUC          0.7992 ± 0.0138
TPR@1%FPR       0.6736 ± 0.0065

Embeddings (768-d CLS) saved to /kaggle/working/nlp/embeddings/
Probabilities saved to /kaggle/working/nlp/probs/


## Cell 10 — Sanity checks

In [10]:
checks_ok = checks_fail = 0
def chk(ok, name, detail=''):
    global checks_ok, checks_fail
    if ok: checks_ok+=1; print(f'[OK]   {name}')
    else:  checks_fail+=1; print(f'[FAIL] {name}  {detail}')

done_s = [s for s in SEEDS if str(s) in all_results]
print('\n=== SANITY CHECKS ===')
chk((OUT_ROOT/'nlp_results.json').exists(), 'nlp_results.json saved')

for seed in done_s:
    for split in ['train','val','test','test_c2']:
        chk((PROBS_DIR/f'{split}_probs_seed{seed}.npy').exists(),     f'Seed {seed} {split} probs')
        chk((EMBEDDINGS_DIR/f'{split}_emb_seed{seed}.npy').exists(),  f'Seed {seed} {split} embs')
    m = all_results[str(seed)]['test_metrics']
    chk(m['auroc']>0.90, f'Seed {seed}: balanced test AUROC>0.90', f'AUROC={m["auroc"]:.4f}')
    # Embedding dimension check
    emb = np.load(EMBEDDINGS_DIR/f'train_emb_seed{seed}.npy')
    chk(emb.shape[1]==768, f'Seed {seed}: emb dim=768', f'shape={emb.shape}')

print(f'\n{checks_ok} passed, {checks_fail} failed')
if checks_fail == 0:
    print('\nALL CHECKS PASSED.')
    print('Next: upload /kaggle/working/nlp/ as Kaggle Dataset "04-nlp-dataset"')
    print('Can run 05_gat_branch.ipynb in parallel while uploading.')
else:
    raise RuntimeError(f'{checks_fail} checks FAILED.')



=== SANITY CHECKS ===
[OK]   nlp_results.json saved
[OK]   Seed 42 train probs
[OK]   Seed 42 train embs
[OK]   Seed 42 val probs
[OK]   Seed 42 val embs
[OK]   Seed 42 test probs
[OK]   Seed 42 test embs
[OK]   Seed 42 test_c2 probs
[OK]   Seed 42 test_c2 embs
[OK]   Seed 42: balanced test AUROC>0.90
[OK]   Seed 42: emb dim=768
[OK]   Seed 1337 train probs
[OK]   Seed 1337 train embs
[OK]   Seed 1337 val probs
[OK]   Seed 1337 val embs
[OK]   Seed 1337 test probs
[OK]   Seed 1337 test embs
[OK]   Seed 1337 test_c2 probs
[OK]   Seed 1337 test_c2 embs
[OK]   Seed 1337: balanced test AUROC>0.90
[OK]   Seed 1337: emb dim=768
[OK]   Seed 2024 train probs
[OK]   Seed 2024 train embs
[OK]   Seed 2024 val probs
[OK]   Seed 2024 val embs
[OK]   Seed 2024 test probs
[OK]   Seed 2024 test embs
[OK]   Seed 2024 test_c2 probs
[OK]   Seed 2024 test_c2 embs
[OK]   Seed 2024: balanced test AUROC>0.90
[OK]   Seed 2024: emb dim=768

31 passed, 0 failed

ALL CHECKS PASSED.
Next: upload /kaggle/working/